# 07 — True FinBERT Inference with T4 Optimization & Parquet Caching

**Goal:** Run real HuggingFace FinBERT (`ProsusAI/finbert`) on the full 31,037-headline corpus, optimized for Colab T4 GPU, with Parquet caching so we never re-score the same CSV twice.

**Optimizations:**
- `batch_size=128` (T4 16GB VRAM fits FinBERT at max_length=512)
- `fp16` inference via `torch.cuda.amp.autocast()` — ~2x speedup
- `max_length=512` (FinBERT's full context window)

**Caching:** SHA256 hash of the source CSV is stored as Parquet metadata. If the source changes, the cache is automatically invalidated.

In [ ]:
import sys, hashlib
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'Data' / 'cryptonews.csv').exists() or (ROOT / 'notebooks').exists():
        break
    if ROOT == ROOT.parent:
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.inference.finbert import (
    cached_score_with_finbert,
    score_with_finbert,
    file_sha256,
    DEFAULT_BATCH_SIZE,
    DEFAULT_MAX_LENGTH,
)

SOURCE_PATH = ROOT / 'Data' / 'cryptonews.csv'
CACHE_PATH  = ROOT / 'Data' / 'cryptews_scored.parquet'

print(f'Source CSV: {SOURCE_PATH}')
print(f'Source SHA256: {file_sha256(SOURCE_PATH)[:16]}...')
print(f'Cache path: {CACHE_PATH}')
print(f'Defaults: batch_size={DEFAULT_BATCH_SIZE}, max_length={DEFAULT_MAX_LENGTH}')

## 7.1 Load the source news CSV
Apply the same Bug Fix 2 (mixed date formats) as in Notebook 01.

In [ ]:
news = pd.read_csv(SOURCE_PATH)
news['date'] = pd.to_datetime(news['date'], format='mixed', utc=True, errors='coerce')
news = news.dropna(subset=['date']).sort_values('date').reset_index(drop=True)
print(f'{len(news):,} headlines | {news.date.min()} → {news.date.max()}')
news.head(2)

## 7.2 GPU check
On Colab T4 the next cell should print `CUDA available: True`. On CPU, the same code runs (just slower).

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    !nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## 7.3 Run cached FinBERT scoring
First run: ~5-7 min on Colab T4 (31k headlines, batch=128, fp16).  
Subsequent runs: <1 sec (cache hit).

In [ ]:
scored = cached_score_with_finbert(
    news_df=news,
    source_path=SOURCE_PATH,
    cache_path=CACHE_PATH,
    force_refresh=False,            # set True to re-run inference
    # T4-optimized settings (defaults already match these):
    model_name='ProsusAI/finbert',
    batch_size=128,
    max_length=512,
    use_fp16=True,
)
scored[['date', 'title', 'llm_sentiment']].head(10)

## 7.4 Inspect the score distribution

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
try:
    fm.fontManager.addfont('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf')
except Exception:
    pass
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)

ax1.hist(scored['llm_sentiment'], bins=50, color='#0f766e', edgecolor='white')
ax1.axvline(0, c='k', ls='--', lw=0.8)
ax1.set_title('FinBERT Sentiment Score Distribution')
ax1.set_xlabel('llm_sentiment (−1 = neg, 0 = neutral, +1 = pos)')
ax1.set_ylabel('Headline count')

# Per-day mean
scored['date_day'] = scored['date'].dt.tz_convert(None).dt.floor('D')
daily = scored.groupby('date_day')['llm_sentiment'].agg(['mean', 'size'])
ax2.plot(daily.index, daily['mean'].rolling(7).mean(), color='#3b82f6', lw=1.5)
ax2.axhline(0, c='k', ls='--', lw=0.8)
ax2.set_title('Daily Mean FinBERT Sentiment (7-day MA)')
ax2.set_xlabel('Date')
ax2.set_ylabel('Mean sentiment')
ax2.tick_params(axis='x', rotation=20)

plt.savefig(ROOT / 'outputs' / 'finbert_score_distribution.png', dpi=140, bbox_inches='tight')
plt.show()

print(f'\nMean sentiment : {scored.llm_sentiment.mean():.4f}')
print(f'Std            : {scored.llm_sentiment.std():.4f}')
print(f'Positive share : {(scored.llm_sentiment > 0.3).mean():.2%}')
print(f'Negative share : {(scored.llm_sentiment < -0.3).mean():.2%}')
print(f'Neutral share  : {((scored.llm_sentiment >= -0.3) & (scored.llm_sentiment <= 0.3)).mean():.2%}')

## 7.5 Verify cache behavior
Run the cell below a second time — it should hit the cache and return in <1 sec.

In [ ]:
scored_cached = cached_score_with_finbert(
    news_df=news,
    source_path=SOURCE_PATH,
    cache_path=CACHE_PATH,
    force_refresh=False,
)
assert scored_cached is not None
print('Cache hit verified.')

## 7.6 Cache invalidation demo
If we modify the source CSV (or pretend to by changing the cache key check), the cache should be invalidated and FinBERT re-runs.

To force re-inference without modifying the CSV, set `force_refresh=True`.

In [ ]:
# Verify cache metadata is persisted
import pandas as pd
meta_check = pd.read_parquet(CACHE_PATH)
print('Cache metadata:')
print(f'  cache_key    : {meta_check.attrs.get("cache_key")}')
print(f'  source_path  : {meta_check.attrs.get("source_path")}')
print(f'  source_sha256: {meta_check.attrs.get("source_sha256", "")[:32]}...')
print(f'  rows         : {len(meta_check):,}')
print(f'  columns      : {list(meta_check.columns)}')

## 7.7 Integrate with the downstream pipeline
Now that we have true FinBERT scores cached, Notebook 02's `merged_with_llm_sentiment.parquet` can be regenerated from this Parquet instead of the TextBlob fallback.

To regenerate the full pipeline outputs with true FinBERT scores:
```bash
python3 scripts/run_pipeline.py  # no --use-precomputed flag
```
The script will check `Data/cryptonews_scored.parquet` first; on cache hit it skips inference entirely.

## 7.8 Summary
- Loaded `ProsusAI/finbert` (finance-tuned BERT) with T4-optimized settings (batch=128, fp16, max_length=512).
- Implemented Parquet caching keyed on the source CSV's SHA256 hash — cache is auto-invalidated if the source changes.
- Cache hit returns in <1 sec; cache miss runs full inference (~5-7 min on T4).
- Output: `Data/cryptonews_scored.parquet` with `llm_sentiment` column in [-1, 1].